# Data Cleaning and Normalization for MOTM Prediction
## Làm sạch và chuẩn hóa dữ liệu cho dự đoán MOTM

### 1. Cài đặt và Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Nếu dùng Google Colab, install thư viện cần thiết
try:
    from google.colab import drive
    drive.mount('/content/drive')
    colab = True
except:
    colab = False
    print("Not running in Google Colab")

print("All libraries imported successfully!")

### 2. Load dữ liệu gốc

In [ ]:
# Đọc file dữ liệu
if colab:
    file_path = '/content/drive/My Drive/MotmPrediction/PlayerCrawl.xlsx'  # Chỉnh path của bạn
else:
    file_path = 'PlayerCrawl.xlsx'

# Load dữ liệu
df = pd.read_excel(file_path)

# Hiển thị thông tin cơ bản
print("=" * 50)
print(f"Tổng số dòng: {len(df)}")
print(f"Tổng số cột: {len(df.columns)}")
print("\nTên các cột:")
print(df.columns.tolist())
print("\n" + "=" * 50)
print("Dữ liệu mẫu:")
print(df.head())
print("\n" + "=" * 50)
print("Thông tin chi tiết:")
print(df.info())
print("\n" + "=" * 50)
print("Thống kê mô tả:")
print(df.describe())

### 3. Kiểm tra giá trị thiếu (Missing Values)

In [ ]:
# Tính tỷ lệ missing values
missing_data = pd.DataFrame({
    'Cột': df.columns,
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing %', ascending=False)

print("Missing Values Analysis:")
print(missing_data)
print("\n" + "=" * 50)

### 4. Kiểm tra dữ liệu trùng lặp (Duplicates)

In [ ]:
# Tìm dữ liệu trùng lặp
print(f"Tổng số dòng trùng lặp: {df.duplicated().sum()}")

if df.duplicated().sum() > 0:
    print("\nDữ liệu trùng lặp:")
    duplicates = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
    print(duplicates.head(10))

### 5. Làm sạch dữ liệu - Định nghĩa các hàm xử lý

In [ ]:
def clean_text(text):
    """Làm sạch dữ liệu text: loại bỏ khoảng trắng dư thừa, định chuẩn hóa"""
    if pd.isna(text):
        return None
    text = str(text).strip()
    text = ' '.join(text.split())  # Loại bỏ khoảng trắng dư thừa
    return text

def clean_numeric(value):
    """Làm sạch dữ liệu số: chuyển đổi thành số, xử lý giá trị không hợp lệ"""
    if pd.isna(value):
        return None
    try:
        # Loại bỏ ký tự không phải số (giữ lại dấu âm và dấu chấm thập phân)
        cleaned = re.sub(r'[^0-9.-]', '', str(value))
        return float(cleaned) if cleaned else None
    except:
        return None

def clean_date(date_value):
    """Làm sạch dữ liệu ngày tháng"""
    if pd.isna(date_value):
        return None
    try:
        if isinstance(date_value, str):
            # Thử các định dạng ngày tháng phổ biến
            for fmt in ["%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%d.%m.%Y"]:
                try:
                    return pd.to_datetime(date_value, format=fmt)
                except:
                    continue
            return pd.to_datetime(date_value)  # Để pandas tự nhận diện
        else:
            return pd.to_datetime(date_value)
    except:
        return None

print("Các hàm làm sạch dữ liệu đã được định nghĩa.")

### 6. Áp dụng các hàm làm sạch

In [ ]:
# Tạo bản sao để làm sạch
df_clean = df.copy()

# HƯỚNG DẪN: Chỉnh sửa đoạn code này theo cấu trúc dữ liệu của bạn
# Ví dụ: nếu có cột 'PlayerName', 'Age', 'Date', 'Rating'...

# Làm sạch các cột text
text_columns = [col for col in df_clean.columns if df_clean[col].dtype == 'object']
for col in text_columns:
    df_clean[col] = df_clean[col].apply(clean_text)

# Làm sạch các cột số (bạn cần điều chỉnh theo tên cột thực tế)
# Ví dụ:
# numeric_columns = ['Age', 'Rating', 'Price']
# for col in numeric_columns:
#     if col in df_clean.columns:
#         df_clean[col] = df_clean[col].apply(clean_numeric)

# Loại bỏ dòng trùng lặp
df_clean = df_clean.drop_duplicates()

print("Dữ liệu đã được làm sạch!")
print(f"\nSố dòng trước: {len(df)}")
print(f"Số dòng sau: {len(df_clean)}")
print(f"Dòng bị xóa: {len(df) - len(df_clean)}")

### 7. Xử lý Missing Values

In [ ]:
# Chiến lược xử lý missing values (điều chỉnh theo nhu cầu):

# Option 1: Loại bỏ dòng có missing values
# df_clean = df_clean.dropna()

# Option 2: Điền giá trị missing bằng giá trị trung bình (cho cột số)
# numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
# df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].mean())

# Option 3: Điền giá trị missing bằng 'Unknown' (cho cột text)
# text_cols = df_clean.select_dtypes(include=['object']).columns
# df_clean[text_cols] = df_clean[text_cols].fillna('Unknown')

# Option 4: Loại bỏ các cột có quá nhiều missing values
# threshold = 0.5  # Nếu cột có > 50% missing, loại bỏ
# cols_to_drop = [col for col in df_clean.columns if df_clean[col].isnull().sum() / len(df_clean) > threshold]
# df_clean = df_clean.drop(columns=cols_to_drop)

print("Missing values sau khi xử lý:")
print(df_clean.isnull().sum())

### 8. Chuẩn hóa dữ liệu (Normalization & Standardization)

### 9. Phát hiện và xử lý Outliers

In [ ]:
def detect_outliers_iqr(df, columns):
    """Phát hiện outliers sử dụng Interquartile Range (IQR)"""
    outliers_mask = pd.Series([False] * len(df))
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        col_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers_mask = outliers_mask | col_outliers
        
        print(f"Cột '{col}':")
        print(f"  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
        print(f"  Khoảng bình thường: [{lower_bound:.2f}, {upper_bound:.2f}]")
        print(f"  Số outliers: {col_outliers.sum()}")
    
    return outliers_mask

# Phát hiện outliers
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    outliers_mask = detect_outliers_iqr(df_clean, numeric_cols)
    print(f"\nTổng số dòng có outliers: {outliers_mask.sum()}")
    
    # Xóa outliers (nếu cần)
    # df_clean = df_clean[~outliers_mask]
    # print(f"Số dòng sau khi xóa outliers: {len(df_clean)}")
else:
    print("Không có cột số để phát hiện outliers.")

### 10. Lưu dữ liệu đã làm sạch

In [ ]:
# Lưu dữ liệu đã làm sạch (bản gốc - chỉ làm sạch)
if colab:
    output_path_clean = '/content/drive/My Drive/MotmPrediction/PlayerCrawl_cleaned.xlsx'
    output_path_normalized = '/content/drive/My Drive/MotmPrediction/PlayerCrawl_normalized.xlsx'
else:
    output_path_clean = 'PlayerCrawl_cleaned.xlsx'
    output_path_normalized = 'PlayerCrawl_normalized.xlsx'

# Lưu dữ liệu đã làm sạch
df_clean.to_excel(output_path_clean, index=False)
print(f"✓ Đã lưu dữ liệu đã làm sạch: {output_path_clean}")

# Lưu dữ liệu đã chuẩn hóa (Min-Max)
df_minmax.to_excel(output_path_normalized, index=False)
print(f"✓ Đã lưu dữ liệu đã chuẩn hóa: {output_path_normalized}")

# Lưu CSV format
csv_clean = output_path_clean.replace('.xlsx', '.csv')
df_clean.to_csv(csv_clean, index=False, encoding='utf-8-sig')
print(f"✓ Đã lưu dữ liệu CSV: {csv_clean}")

### 11. Báo cáo tóm tắt

In [ ]:
print("\n" + "="*60)
print("BÁO CÁO TÓM TẮT QUÁ TRÌNH LÀM SẠCH DỮ LIỆU")
print("="*60)

print(f"\n1. THÔNG TIN TỔNG QUÁT:")
print(f"   - Số dòng gốc: {len(df)}")
print(f"   - Số dòng sau làm sạch: {len(df_clean)}")
print(f"   - Dòng bị xóa: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.2f}%)")
print(f"   - Số cột: {len(df_clean.columns)}")

print(f"\n2. CÁC BƯỚC THỰC HIỆN:")
print(f"   ✓ Loại bỏ khoảng trắng dư thừa (text)")
print(f"   ✓ Loại bỏ dòng trùng lặp: {len(df) - len(df_clean)} dòng")
print(f"   ✓ Phân tích missing values")
print(f"   ✓ Chuẩn hóa dữ liệu số (Min-Max Scaling)")
print(f"   ✓ Phát hiện outliers (IQR method)")

print(f"\n3. KIỂU DỮ LIỆU:")
print(df_clean.dtypes)

print(f"\n4. ĐỐI TƯỢNG DỮ LIỆU MẪU:")
print(df_clean.head())

print(f"\n5. FILE ĐẦU RA:")
print(f"   - Dữ liệu đã làm sạch: PlayerCrawl_cleaned.xlsx")
print(f"   - Dữ liệu đã chuẩn hóa: PlayerCrawl_normalized.xlsx")
print(f"   - CSV format: PlayerCrawl_cleaned.csv")

print("\n" + "="*60)
print("Quá trình làm sạch và chuẩn hóa dữ liệu hoàn tất!")
print("="*60)